# BE277 – Business Analytics for Managers and Entrepreneurs
## Zomato Restaurant Dataset Analysis
**University of Essex | Essex Business School | Spring 2026**

---

| Item | Detail |
|---|---|
| Module | BE277 – Business Analytics for Managers and Entrepreneurs |
| Assessment | Individual Report (100%) |
| Dataset | zomato.csv + Country_Code.xlsx |

---

### Notebook Structure
1. Setup & Imports  
2. Data Loading and Cleaning  
3. Descriptive Analytics  
   - 3.1 Geographic Distribution  
   - 3.2 Cuisine Insights  
   - 3.3 Ratings and Engagement  
   - 3.4 Pricing and Value  
   - 3.5 Online Delivery  
   - 3.6 Correlation Analysis  
4. Predictive Analytics  
   - 4.1 Rating Prediction Models  
   - 4.2 K-Means Clustering  
5. Summary of Findings


## 1. Setup & Imports

In [ ]:
# Install required libraries (run once, then restart kernel)
# !pip install pandas numpy matplotlib seaborn scikit-learn openpyxl

import os
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                              r2_score, silhouette_score)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings('ignore')

# Show plots inline in the notebook
%matplotlib inline

print("All libraries imported successfully.")

In [ ]:
# ── Colour palette (consistent across all figures) ───────────────────────────
C1   = '#003f7f'   # deep navy
C2   = '#c0392b'   # crimson
C3   = '#2980b9'   # mid-blue
C4   = '#27ae60'   # green
C5   = '#8e44ad'   # purple
C6   = '#e67e22'   # orange
GREY = '#95a5a6'
PALETTE = [C1, C2, C3, C4, C5, C6, '#1abc9c', '#e74c3c']

# ── Global matplotlib style ───────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    '#fafafa',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.edgecolor':    '#cccccc',
    'axes.linewidth':    0.8,
    'axes.grid':         True,
    'grid.alpha':        0.4,
    'grid.linestyle':    '--',
    'grid.linewidth':    0.5,
    'font.family':       'DejaVu Sans',
    'axes.titlesize':    12,
    'axes.titleweight':  'bold',
    'axes.labelsize':    10,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'legend.fontsize':   9,
    'legend.framealpha': 0.8,
})

print("Palette and style configured.")

## 2. Data Loading and Cleaning

In [ ]:
# ── FILE PATHS: update these two lines to match your machine ─────────────────
ZOMATO_PATH  = r'D:\zomato-business-analytics\data\zomato.csv'
COUNTRY_PATH = r'D:\zomato-business-analytics\data\Country_Code.xlsx'
# ─────────────────────────────────────────────────────────────────────────────

# Load datasets
df = pd.read_csv(ZOMATO_PATH, encoding='latin-1')
cc = pd.read_excel(COUNTRY_PATH)

print("Zomato shape  :", df.shape)
print("Country codes :", cc.shape)
df.head(3)

In [ ]:
# Merge country names onto main dataset
df = df.merge(cc, on='Country Code', how='left')

# Drop the 9 rows with missing Cuisines (less than 0.1% of data)
df = df.dropna(subset=['Cuisines']).reset_index(drop=True)

# Encode binary service flags as integers for modelling
df['Online_Delivery'] = (df['Has Online delivery'] == 'Yes').astype(int)
df['Table_Booking']   = (df['Has Table booking']   == 'Yes').astype(int)

# Rated subset: exclude restaurants with Aggregate rating == 0 (Not rated)
df_rated = df[df['Aggregate rating'] > 0].copy()

print("Total records  (after cleaning) :", len(df))
print("Rated records  (rating > 0)     :", len(df_rated))
print("Countries                        :", df['Country'].nunique())
print("Cities                           :", df['City'].nunique())
print("Missing values remaining         :", df.isnull().sum().sum())

In [ ]:
# Descriptive statistics of key numeric variables
df_rated[['Aggregate rating', 'Votes', 'Average Cost for two', 'Price range']].describe().round(2)

In [ ]:
# Value counts for categorical service variables
print("Online Delivery distribution:")
print(df['Has Online delivery'].value_counts())
print()
print("Table Booking distribution:")
print(df['Has Table booking'].value_counts())
print()
print("Rating text distribution:")
print(df['Rating text'].value_counts())

## 3. Descriptive Analytics

Descriptive analytics summarises the key characteristics of the dataset using
visualisations, frequency distributions, and correlation statistics.


### 3.1 Geographic Distribution

In [ ]:
# Country distribution summary
country_counts = df['Country'].value_counts()
print("Top 5 countries:")
print(country_counts.head())
print()
india_pct = country_counts['India'] / len(df) * 100
print("India share: {:.1f}%".format(india_pct))

In [ ]:
# Figure 1 - Geographic Distribution
country_counts = df['Country'].value_counts()
top_other = country_counts[:6].copy()
top_other['Other'] = country_counts[6:].sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Figure 1: Geographic Distribution of Restaurants',
             fontsize=13, fontweight='bold', y=1.01)

# Bar chart
bar_colors = [C1 if c == 'India' else C3 for c in country_counts.index[::-1]]
axes[0].barh(country_counts.index[::-1], country_counts.values[::-1],
             color=bar_colors, edgecolor='white', height=0.7)
axes[0].set_title('(a) Restaurant Count by Country')
axes[0].set_xlabel('Number of Restaurants')
for i, v in enumerate(country_counts.values[::-1]):
    axes[0].text(v + 20, i, '{:,}'.format(v), va='center', fontsize=8)

# Pie chart
wedge_colors = [C1] + [C3]*4 + [C4, GREY]
axes[1].pie(top_other.values, labels=top_other.index, colors=wedge_colors,
            autopct='%1.1f%%', startangle=140,
            wedgeprops=dict(edgecolor='white', linewidth=1.2),
            textprops={'fontsize': 8.5})
axes[1].set_title('(b) Proportional Market Share')

plt.tight_layout()
plt.show()

In [ ]:
# Figure 2 - Top 10 Cities
top_cities = df['City'].value_counts().head(10)
colors_c = [C2 if i == 0 else C1 if i < 3 else C3 for i in range(len(top_cities))]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(top_cities.index, top_cities.values, color=colors_c,
              edgecolor='white', width=0.7)
ax.set_title('Figure 2: Top 10 Cities by Number of Listed Restaurants')
ax.set_xlabel('City')
ax.set_ylabel('Number of Restaurants')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 30,
            '{:,}'.format(int(bar.get_height())),
            ha='center', fontsize=8, fontweight='bold')
plt.xticks(rotation=30, ha='right')
patches = [mpatches.Patch(color=C2, label='Rank 1'),
           mpatches.Patch(color=C1, label='Rank 2-3'),
           mpatches.Patch(color=C3, label='Rank 4-10')]
ax.legend(handles=patches, loc='upper right')
plt.tight_layout()
plt.show()

### 3.2 Cuisine and Menu Insights

In [ ]:
# Extract individual cuisines from the comma-separated Cuisines field
cuisine_list = []
for entry in df['Cuisines'].dropna():
    cuisine_list.extend([c.strip() for c in entry.split(',')])

cuisine_series = pd.Series(Counter(cuisine_list))
top15 = cuisine_series.nlargest(15)

print("Total unique cuisines:", cuisine_series.nunique())
print()
print("Top 5 cuisines:")
print(top15.head())

In [ ]:
# Figure 4 - Top 15 Cuisines
colors_c = [C2 if i < 3 else C1 if i < 7 else C3 for i in range(len(top15))]

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(top15.index[::-1], top15.values[::-1],
        color=colors_c[::-1], edgecolor='white', height=0.7)
ax.set_title('Figure 4: Top 15 Most Listed Cuisines on Zomato')
ax.set_xlabel('Number of Restaurant Listings')
for i, v in enumerate(top15.values[::-1]):
    ax.text(v + 15, i, '{:,}'.format(int(v)), va='center', fontsize=8.5)
patches = [mpatches.Patch(color=C2, label='Top 3'),
           mpatches.Patch(color=C1, label='Rank 4-7'),
           mpatches.Patch(color=C3, label='Rank 8-15')]
ax.legend(handles=patches, loc='lower right')
plt.tight_layout()
plt.show()

### 3.3 Ratings and Engagement

In [ ]:
# Summary statistics for ratings
print("Rated restaurants: {:,}".format(len(df_rated)))
print("Mean rating  : {:.2f}".format(df_rated['Aggregate rating'].mean()))
print("Median rating: {:.2f}".format(df_rated['Aggregate rating'].median()))
print("Std dev      : {:.2f}".format(df_rated['Aggregate rating'].std()))
print()
print("Not rated (rating == 0):", (df['Aggregate rating'] == 0).sum())
print("% Not rated            : {:.1f}%".format(
    (df['Aggregate rating'] == 0).sum() / len(df) * 100))

In [ ]:
# Figure 3 - Ratings and Engagement (3 panels)
rating_order = ['Poor', 'Average', 'Good', 'Very Good', 'Excellent', 'Not rated']
rc = df['Rating text'].value_counts().reindex(rating_order).fillna(0)
bar_colors_rt = [C2, C6, GREY, C3, C4, '#bdc3c7']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Figure 3: Ratings and Engagement Analysis', fontsize=13, fontweight='bold')

# (a) Histogram
axes[0].hist(df_rated['Aggregate rating'], bins=24, color=C3, edgecolor='white', alpha=0.85)
axes[0].axvline(df_rated['Aggregate rating'].mean(), color=C2, linestyle='--', linewidth=2,
                label='Mean = {:.2f}'.format(df_rated['Aggregate rating'].mean()))
axes[0].axvline(df_rated['Aggregate rating'].median(), color=C4, linestyle=':', linewidth=2,
                label='Median = {:.2f}'.format(df_rated['Aggregate rating'].median()))
axes[0].set_title('(a) Rating Distribution')
axes[0].set_xlabel('Aggregate Rating')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# (b) Category bar chart
axes[1].bar(rc.index, rc.values, color=bar_colors_rt, edgecolor='white', width=0.7)
for i, v in enumerate(rc.values):
    axes[1].text(i, v + 30, '{:,}'.format(int(v)), ha='center', fontsize=8)
axes[1].set_title('(b) Rating Category Counts')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Count')
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=25, ha='right')

# (c) Votes distribution
axes[2].hist(df_rated['Votes'].clip(upper=2000), bins=40, color=C5, edgecolor='white', alpha=0.85)
median_v = df_rated['Votes'].median()
axes[2].axvline(median_v, color=C2, linestyle='--', linewidth=2,
                label='Median = {:.0f}'.format(median_v))
axes[2].set_title('(c) Votes Distribution (capped at 2,000)')
axes[2].set_xlabel('Votes')
axes[2].set_ylabel('Frequency')
axes[2].legend()

plt.tight_layout()
plt.show()

### 3.4 Pricing and Value Analysis

In [ ]:
# Price range summary
price_labels = {1: 'Budget', 2: 'Mid-range', 3: 'Premium', 4: 'Fine Dining'}
price_counts = df['Price range'].value_counts().sort_index()
mean_by_price = df_rated.groupby('Price range')['Aggregate rating'].mean().round(2)

summary = pd.DataFrame({
    'Label':       [price_labels[i] for i in price_counts.index],
    'Count':       price_counts.values,
    'Share (%)':   (price_counts.values / len(df) * 100).round(1),
    'Avg Rating':  mean_by_price.values
})
summary.index = price_counts.index
summary

In [ ]:
# Figure 5 - Pricing Analysis (3 panels)
price_label_list = ['Budget (1)', 'Mid-range (2)', 'Premium (3)', 'Fine Dining (4)']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Figure 5: Pricing, Value, and Rating Relationship',
             fontsize=13, fontweight='bold')

# (a) Count per tier
pr_counts = df['Price range'].value_counts().sort_index()
axes[0].bar(price_label_list, pr_counts.values, color=PALETTE[:4], edgecolor='white', width=0.6)
for i, v in enumerate(pr_counts.values):
    axes[0].text(i, v + 40, '{:,}'.format(v), ha='center', fontsize=9, fontweight='bold')
axes[0].set_title('(a) Count by Price Tier')
axes[0].set_ylabel('Number of Restaurants')
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=15)

# (b) Box plot of ratings by tier
groups = [df_rated[df_rated['Price range'] == i]['Aggregate rating'] for i in [1, 2, 3, 4]]
bp = axes[1].boxplot(groups, patch_artist=True, notch=False,
                      labels=['Budget', 'Mid', 'Premium', 'Fine'],
                      medianprops=dict(color='white', linewidth=2))
for patch, color in zip(bp['boxes'], PALETTE[:4]):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)
axes[1].set_title('(b) Rating Distribution by Price Tier')
axes[1].set_xlabel('Price Tier')
axes[1].set_ylabel('Aggregate Rating')

# (c) Mean rating with error bars
mean_r = df_rated.groupby('Price range')['Aggregate rating'].mean()
std_r  = df_rated.groupby('Price range')['Aggregate rating'].std()
axes[2].bar(price_label_list, mean_r.values, yerr=std_r.values,
            color=PALETTE[:4], edgecolor='white', capsize=5, width=0.6)
for i, v in enumerate(mean_r.values):
    axes[2].text(i, v + 0.05, '{:.2f}'.format(v), ha='center', fontsize=10, fontweight='bold')
axes[2].set_title('(c) Mean Rating +/- Std Dev by Price Tier')
axes[2].set_ylabel('Mean Rating')
axes[2].set_ylim(0, 5.2)
plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=15)

plt.tight_layout()
plt.show()

### 3.5 Online Delivery and Services

In [ ]:
# Online delivery statistics
del_counts = df['Has Online delivery'].value_counts()
del_avg = df_rated.groupby('Has Online delivery')['Aggregate rating'].mean()

print("Online delivery counts:")
print(del_counts)
print()
print("Average rating by delivery status:")
print(del_avg.round(3))
print()
print("Rating difference: +{:.3f}".format(del_avg['Yes'] - del_avg['No']))

In [ ]:
# Figure 6 - Online Delivery Analysis (3 panels)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Figure 6: Online Delivery and Services Analysis',
             fontsize=13, fontweight='bold')

# (a) Pie chart
axes[0].pie(del_counts.values, labels=['No Delivery', 'Has Delivery'],
            colors=[GREY, C2], autopct='%1.1f%%', startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[0].set_title('(a) Online Delivery Availability')

# (b) Mean rating comparison
del_avg_full = df_rated.groupby('Has Online delivery')['Aggregate rating'].agg(['mean', 'std'])
bars = axes[1].bar(['No Delivery', 'Has Delivery'],
                   del_avg_full['mean'].values,
                   yerr=del_avg_full['std'].values,
                   color=[GREY, C2], edgecolor='white', capsize=6, width=0.5)
for bar, v in zip(bars, del_avg_full['mean'].values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.04,
                 '{:.3f}'.format(v), ha='center', fontsize=12, fontweight='bold')
axes[1].set_title('(b) Average Rating: Delivery vs No Delivery')
axes[1].set_ylabel('Average Aggregate Rating')
axes[1].set_ylim(0, 5)

# (c) Grouped bar by top 5 countries
top5 = df['Country'].value_counts().head(5).index
sub  = df_rated[df_rated['Country'].isin(top5)]
pivot = sub.groupby(['Country', 'Has Online delivery'])['Aggregate rating'].mean().unstack()
pivot.columns = ['No Delivery', 'Has Delivery']
x = np.arange(len(pivot))
w = 0.35
axes[2].bar(x - w/2, pivot['No Delivery'], w, label='No Delivery', color=GREY, edgecolor='white')
axes[2].bar(x + w/2, pivot['Has Delivery'], w, label='Has Delivery', color=C2, edgecolor='white')
axes[2].set_xticks(x)
axes[2].set_xticklabels(pivot.index, rotation=20, ha='right')
axes[2].set_title('(c) Delivery Impact by Top 5 Countries')
axes[2].set_ylabel('Average Rating')
axes[2].legend()

plt.tight_layout()
plt.show()

### 3.6 Correlation Analysis

In [ ]:
# Compute Pearson correlation matrix
corr_cols = ['Aggregate rating', 'Votes', 'Average Cost for two',
             'Price range', 'Online_Delivery', 'Table_Booking']
corr_mat = df_rated[corr_cols].corr().round(3)
print("Correlation Matrix:")
corr_mat

In [ ]:
# Figure 7 - Correlation Heatmap
display_labels = ['Rating', 'Votes', 'Cost/2', 'Price Range', 'Online Del.', 'Table Book.']
mask = np.triu(np.ones_like(corr_mat, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.8, ax=ax, mask=mask,
            cbar_kws={'shrink': 0.8, 'label': 'Pearson r'},
            xticklabels=display_labels, yticklabels=display_labels,
            annot_kws={'size': 11, 'weight': 'bold'})
ax.set_title('Figure 7: Pearson Correlation Matrix - Key Variables', pad=14, fontsize=13)
plt.tight_layout()
plt.show()

print("Key correlations:")
print("  Votes vs Rating      : {:.3f}".format(corr_mat.loc['Aggregate rating', 'Votes']))
print("  Price Range vs Rating: {:.3f}".format(corr_mat.loc['Aggregate rating', 'Price range']))
print("  Online Del. vs Rating: {:.3f}".format(corr_mat.loc['Aggregate rating', 'Online_Delivery']))

## 4. Predictive Analytics

Two predictive techniques are applied:
1. **Supervised Regression** – three models compared (Linear Regression, Random Forest, Gradient Boosting) to predict Aggregate Rating  
2. **Unsupervised Clustering** – K-Means to segment restaurants into strategic archetypes


### 4.1 Rating Prediction Models

In [ ]:
# Prepare features and target variable
le = LabelEncoder()
df_rated['Country_enc'] = le.fit_transform(df_rated['Country'])

FEATURES = ['Price range', 'Average Cost for two', 'Votes',
            'Online_Delivery', 'Table_Booking', 'Country_enc']
TARGET   = 'Aggregate rating'

X = df_rated[FEATURES]
y = df_rated[TARGET]

# 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Feature set    :", FEATURES)
print("Training rows  : {:,}".format(len(X_train)))
print("Test rows      : {:,}".format(len(X_test)))
X.describe().round(2)

In [ ]:
# Train all three models and collect metrics
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te, X_all, y_all, cv=5):
    model.fit(X_tr, y_tr)
    y_pred    = model.predict(X_te)
    r2        = r2_score(y_te, y_pred)
    mae       = mean_absolute_error(y_te, y_pred)
    rmse      = np.sqrt(mean_squared_error(y_te, y_pred))
    cv_scores = cross_val_score(model, X_all, y_all, cv=cv, scoring='r2')
    return {
        'Model': name, 'R2': round(r2, 4), 'MAE': round(mae, 4), 'RMSE': round(rmse, 4),
        'CV R2 Mean': round(cv_scores.mean(), 4), 'CV R2 Std': round(cv_scores.std(), 4),
        'y_pred': y_pred, 'model': model, 'cv_scores': cv_scores
    }

print("Training models (this may take ~60 seconds)...")

lr_res = evaluate_model('Linear Regression', LinearRegression(),
                         X_train, y_train, X_test, y_test, X, y)
rf_res = evaluate_model('Random Forest',
                         RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1),
                         X_train, y_train, X_test, y_test, X, y)
gb_res = evaluate_model('Gradient Boosting',
                         GradientBoostingRegressor(n_estimators=150, max_depth=4, random_state=42),
                         X_train, y_train, X_test, y_test, X, y)

results = [lr_res, rf_res, gb_res]

# Print performance table
perf_df = pd.DataFrame([{k: v for k, v in r.items()
                          if k not in ('y_pred', 'model', 'cv_scores')}
                         for r in results]).set_index('Model')
print()
print("Model Performance:")
perf_df

In [ ]:
# Figure 8 - Model Evaluation Dashboard (5 panels)
names = [r['Model'] for r in results]
r2s   = [r['R2']   for r in results]
maes  = [r['MAE']  for r in results]
rmses = [r['RMSE'] for r in results]

rf_model = rf_res['model']
gb_model = gb_res['model']

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)
fig.suptitle('Figure 8: Predictive Model Evaluation Dashboard',
             fontsize=14, fontweight='bold', y=1.01)

# (a) R2 bar
ax0 = fig.add_subplot(gs[0, 0])
bars = ax0.bar(names, r2s, color=[GREY, C3, C2], edgecolor='white', width=0.55)
for bar, v in zip(bars, r2s):
    ax0.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.008,
             '{:.3f}'.format(v), ha='center', fontsize=10, fontweight='bold')
ax0.set_title('(a) Model Comparison - R2 Score')
ax0.set_ylabel('R2 Score')
ax0.set_ylim(0, 0.75)
plt.setp(ax0.xaxis.get_majorticklabels(), rotation=12, ha='right', fontsize=8.5)

# (b) MAE and RMSE
ax1 = fig.add_subplot(gs[0, 1])
x = np.arange(len(names))
w = 0.35
ax1.bar(x - w/2, maes,  w, label='MAE',  color=C3, edgecolor='white')
ax1.bar(x + w/2, rmses, w, label='RMSE', color=C5, edgecolor='white')
ax1.set_xticks(x)
ax1.set_xticklabels(names, rotation=12, ha='right', fontsize=8.5)
ax1.set_title('(b) Error Metrics by Model')
ax1.set_ylabel('Error (rating scale)')
ax1.legend()

# (c) Cross-validation box (RF)
ax2 = fig.add_subplot(gs[0, 2])
cv_s = rf_res['cv_scores']
ax2.boxplot(cv_s, patch_artist=True,
            boxprops=dict(facecolor=C3, alpha=0.6),
            medianprops=dict(color=C2, linewidth=2))
for s in cv_s:
    ax2.scatter(1, s, color=C2, zorder=5, s=40, alpha=0.8)
ax2.set_title('(c) RF 5-Fold Cross-Validation R2')
ax2.set_ylabel('R2 Score')
ax2.set_ylim(0, 1)
ax2.set_xticklabels(['RF Cross-Val'])

# (d) Feature importance (RF)
feat_labels = ['Price Range', 'Avg Cost/2', 'Votes', 'Online Delivery', 'Table Booking', 'Country']
ax3 = fig.add_subplot(gs[1, 0:2])
imp = rf_model.feature_importances_
idx = np.argsort(imp)
imp_colors = [C2 if imp[i] == max(imp) else C1 if imp[i] > np.mean(imp) else C3 for i in idx]
ax3.barh([feat_labels[i] for i in idx], imp[idx], color=imp_colors, edgecolor='white', height=0.6)
ax3.set_title('(d) Feature Importance - Random Forest')
ax3.set_xlabel('Importance Score')
for i, v in enumerate(imp[idx]):
    ax3.text(v + 0.002, i, '{:.3f}'.format(v), va='center', fontsize=9)

# (e) Actual vs Predicted (Gradient Boosting)
ax4 = fig.add_subplot(gs[1, 2])
yp_gb = gb_model.predict(X_test)
ax4.scatter(y_test, yp_gb, alpha=0.25, color=C2, s=12, label='Predictions')
lims = [min(y_test.min(), yp_gb.min()), max(y_test.max(), yp_gb.max())]
ax4.plot(lims, lims, color=C1, linewidth=2, label='Perfect Fit')
ax4.set_title('(e) Actual vs Predicted - Gradient Boosting')
ax4.set_xlabel('Actual Rating')
ax4.set_ylabel('Predicted Rating')
ax4.text(0.05, 0.90, 'R2 = {:.3f}\nMAE = {:.3f}'.format(gb_res['R2'], gb_res['MAE']),
         transform=ax4.transAxes,
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.85), fontsize=9)
ax4.legend(fontsize=8)

plt.tight_layout()
plt.show()

### 4.2 K-Means Restaurant Clustering

In [ ]:
# Prepare clustering data (standardise features)
cluster_features = ['Price range', 'Aggregate rating', 'Votes', 'Online_Delivery']
Xc = df_rated[cluster_features].copy()

scaler = StandardScaler()
Xs = scaler.fit_transform(Xc)

# Elbow method and silhouette scores for k = 2 to 8
inertias, sil_scores = [], []
for k in range(2, 9):
    km  = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(Xs)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(Xs, lbl))

sil_df = pd.DataFrame({'k': range(2, 9), 'Inertia': inertias, 'Silhouette': sil_scores})
print("Elbow and silhouette scores:")
sil_df

In [ ]:
# Fit final K-Means model with k=4
K_OPTIMAL = 4

km_final = KMeans(n_clusters=K_OPTIMAL, random_state=42, n_init=10)
labels   = km_final.fit_predict(Xs)
Xc['Cluster'] = labels

# PCA projection for 2D visualisation
pca    = PCA(n_components=2)
Xpca   = pca.fit_transform(Xs)
exp_var = pca.explained_variance_ratio_

# Cluster profile summary
cp = Xc.groupby('Cluster')[cluster_features].mean().round(2)
print("Cluster Profiles:")
cp

In [ ]:
# Figure 9 - Clustering Analysis (4 panels)
cluster_palette = [C1, C2, C3, C4]
feat_names = ['Price Range', 'Avg Rating', 'Votes', 'Online Del.']

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('Figure 9: K-Means Clustering Analysis (k={})'.format(K_OPTIMAL),
             fontsize=14, fontweight='bold')

# (a) Elbow
axes[0, 0].plot(range(2, 9), inertias, 'o-', color=C1, linewidth=2, markersize=8)
axes[0, 0].axvline(K_OPTIMAL, color=C2, linestyle='--', linewidth=1.5,
                   label='Selected k={}'.format(K_OPTIMAL))
axes[0, 0].set_title('(a) Elbow Method - Inertia vs k')
axes[0, 0].set_xlabel('k')
axes[0, 0].set_ylabel('Inertia (SSE)')
axes[0, 0].legend()
axes[0, 0].set_xticks(range(2, 9))

# (b) Silhouette
axes[0, 1].plot(range(2, 9), sil_scores, 's-', color=C4, linewidth=2, markersize=8)
axes[0, 1].axvline(K_OPTIMAL, color=C2, linestyle='--', linewidth=1.5,
                   label='Selected k={}'.format(K_OPTIMAL))
axes[0, 1].set_title('(b) Silhouette Score vs k')
axes[0, 1].set_xlabel('k')
axes[0, 1].set_ylabel('Silhouette Score')
axes[0, 1].legend()
axes[0, 1].set_xticks(range(2, 9))

# (c) PCA scatter
for cl in range(K_OPTIMAL):
    mask = labels == cl
    axes[1, 0].scatter(Xpca[mask, 0], Xpca[mask, 1],
                        c=cluster_palette[cl], alpha=0.4,
                        s=15, label='Cluster {}'.format(cl))
axes[1, 0].set_title('(c) Clusters in PCA Space (PC1={:.1f}%, PC2={:.1f}%)'.format(
    exp_var[0]*100, exp_var[1]*100))
axes[1, 0].set_xlabel('Principal Component 1')
axes[1, 0].set_ylabel('Principal Component 2')
axes[1, 0].legend(markerscale=2)

# (d) Normalised profile bar chart
cp_num  = cp.values
cp_norm = (cp_num - cp_num.min(axis=0)) / (cp_num.max(axis=0) - cp_num.min(axis=0) + 1e-9)
x = np.arange(len(feat_names))
w = 0.2
for i in range(K_OPTIMAL):
    axes[1, 1].bar(x + i * w, cp_norm[i], w,
                   label='Cluster {}'.format(i),
                   color=cluster_palette[i], edgecolor='white', alpha=0.85)
axes[1, 1].set_xticks(x + w * 1.5)
axes[1, 1].set_xticklabels(feat_names, rotation=15, ha='right')
axes[1, 1].set_title('(d) Normalised Cluster Profiles')
axes[1, 1].set_ylabel('Normalised Score (0-1)')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Cluster archetype labelling and interpretation table
archetypes = {
    0: 'Budget, Low-Engagement',
    1: 'Budget, Delivery-Enabled',
    2: 'Premium, High-Engagement',
    3: 'Premium, Niche/Specialist'
}
cluster_sizes = Xc['Cluster'].value_counts().sort_index()

interp = cp.copy()
interp['Archetype'] = archetypes.values()
interp['Size']      = cluster_sizes.values
interp['Online Del.'] = interp['Online_Delivery'].apply(
    lambda x: 'Yes' if x > 0.5 else 'Partial' if x > 0.1 else 'Rare')
interp = interp.drop(columns=['Online_Delivery'])

print("Cluster Summary Table:")
interp

## 5. Summary of Findings

In [ ]:
print("=" * 60)
print("SUMMARY OF KEY FINDINGS")
print("=" * 60)

print("\n[DATASET]")
print("  Records          : {:,}".format(len(df)))
print("  Rated subset     : {:,}".format(len(df_rated)))
print("  Countries        : {}".format(df['Country'].nunique()))

print("\n[DESCRIPTIVE]")
print("  India share      : {:.1f}%".format(
    df['Country'].value_counts()['India'] / len(df) * 100))
print("  Mean rating      : {:.2f} / 5.0".format(df_rated['Aggregate rating'].mean()))
print("  Not rated        : {:.1f}% of all listings".format(
    (df['Aggregate rating'] == 0).sum() / len(df) * 100))
print("  Delivery premium : +{:.3f} avg rating".format(
    df_rated.groupby('Has Online delivery')['Aggregate rating'].mean()['Yes'] -
    df_rated.groupby('Has Online delivery')['Aggregate rating'].mean()['No']))

print("\n[PREDICTIVE MODELS]")
for r in results:
    print("  {:<22}  R2={:.4f}  MAE={:.4f}  RMSE={:.4f}".format(
        r['Model'], r['R2'], r['MAE'], r['RMSE']))

best = max(results, key=lambda r: r['R2'])
print("  Best model: {} (R2={:.4f})".format(best['Model'], best['R2']))

print("\n[CLUSTERING]")
for cl in range(4):
    print("  Cluster {}  {:30s}  Rating={:.2f}  Votes={:.0f}".format(
        cl, archetypes[cl],
        cp.loc[cl, 'Aggregate rating'],
        cp.loc[cl, 'Votes']))

print("\n" + "=" * 60)
print("All analysis complete. Figures rendered inline above.")
print("=" * 60)